# 04 — Pipeline TTS

ElevenLabs, OpenAI, Cartesia, LMNT, Rime.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
%load_ext autoreload
%autoreload 2

import _setup
env = _setup.load()
print(f'getpatter version target: {env.patter_version}')


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


In [ ]:
import sys
import getpatter
with _setup.cell('version_check', tier=1, env=env) as ok:
    if ok:
        print(f'getpatter {getpatter.__version__} on Python {sys.version.split()[0]}')
        assert getpatter.__version__ >= env.patter_version, \
            f'installed {getpatter.__version__} < target {env.patter_version}'


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


In [ ]:
from getpatter import Patter, Twilio
with _setup.cell('local_mode', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(
                account_sid='ACtest00000000000000000000000000',
                auth_token='test',
            ),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        # Verify carrier is wired up via LocalConfig.
        assert p._local_config.telephony_provider == 'twilio'
        assert p._local_config.phone_number == '+15555550100'
        print(f'provider={p._local_config.telephony_provider}  phone={p._local_config.phone_number}')


### Cloud mode (coming soon)
When `api_key=` is supported, Patter cloud handles telephony. For now, the SDK raises `NotImplementedError` — this cell verifies the guard.


In [ ]:
from getpatter import Patter
with _setup.cell('cloud_mode', tier=1, env=env) as ok:
    if ok:
        try:
            Patter(api_key='pt_test_xxx')
            raise AssertionError('expected NotImplementedError — cloud mode guard missing')
        except NotImplementedError as exc:
            print(f'cloud mode guard OK: {exc}')


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the engine from `engine=` / `stt=`/`tts=`.


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI
with _setup.cell('agent_engines', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid='ACtest00000000000000000000000000',
                           auth_token='test'),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        rt = p.agent(system_prompt='hi', engine=OpenAIRealtime(api_key='sk-test'))
        cv = p.agent(system_prompt='hi', engine=ElevenLabsConvAI(api_key='el-test', agent_id='a1'))
        pl = p.agent(system_prompt='hi')  # default: OpenAI Realtime fallback
        assert rt.provider == 'openai_realtime', rt.provider
        assert cv.provider == 'elevenlabs_convai', cv.provider
        assert pl.provider == 'openai_realtime', pl.provider
        print(f'rt.provider={rt.provider}  cv.provider={cv.provider}  pl.provider={pl.provider}')


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


### Cloud mode
Same SDK, just an `api_key=` instead of a carrier — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the mode from `engine=` / `stt=`/`tts=`.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises TTS provider construction and (T3) live synthesis.


### TTS provider construction


In [ ]:
from getpatter import ElevenLabsTTS, OpenAITTS, CartesiaTTS, RimeTTS, LMNTTTS
with _setup.cell('tts_providers', tier=1, env=env) as ok:
    if ok:
        el = ElevenLabsTTS(api_key='el-test', voice_id='21m00Tcm4TlvDq8ikWAM')
        ot = OpenAITTS(api_key='sk-test', voice='alloy', model='tts-1')
        ca = CartesiaTTS(api_key='ca-test')
        ri = RimeTTS(api_key='ri-test')
        lm = LMNTTTS(api_key='lm-test')
        for name, provider in [('ElevenLabs', el), ('OpenAI', ot), ('Cartesia', ca), ('Rime', ri), ('LMNT', lm)]:
            print(f'{name}: {type(provider).__name__}')
        assert el.voice_id == '21m00Tcm4TlvDq8ikWAM'
        assert ot.voice == 'alloy'


### TTS text preparation


In [ ]:
from getpatter import filter_for_tts
with _setup.cell('tts_text_prep', tier=1, env=env) as ok:
    if ok:
        samples = [
            '**Bold** and *italic* text.',
            'Visit https://example.com for more. 🎉',
            'Call us at +1 (800) 555-0100.',
            'Code: `x = 1 + 2`',
        ]
        for raw in samples:
            clean = filter_for_tts(raw)
            print(f'  in:  {raw}')
            print(f'  out: {clean}')
            print()


### Live: ElevenLabs TTS synthesis  *(T3 — requires `ELEVENLABS_API_KEY`)*

Synthesises a short phrase and reports the audio byte count.


In [ ]:
import httpx
with _setup.cell('elevenlabs_tts_live', tier=3, required=['elevenlabs_key'], env=env) as ok:
    if ok:
        voice_id = '21m00Tcm4TlvDq8ikWAM'  # Rachel (default)
        resp = httpx.post(
            f'https://api.elevenlabs.io/v1/text-to-speech/{voice_id}',
            headers={
                'xi-api-key': env.elevenlabs_key,
                'Content-Type': 'application/json',
            },
            json={'text': 'Hello from Patter.', 'model_id': 'eleven_monolingual_v1'},
            timeout=30,
        )
        resp.raise_for_status()
        print(f'ElevenLabs audio: {len(resp.content)} bytes  content-type: {resp.headers["content-type"]}')
        assert len(resp.content) > 0


### Live: OpenAI TTS synthesis  *(T3 — requires `OPENAI_API_KEY`)*


In [ ]:
import httpx
with _setup.cell('openai_tts_live', tier=3, required=['openai_key'], env=env) as ok:
    if ok:
        resp = httpx.post(
            'https://api.openai.com/v1/audio/speech',
            headers={
                'Authorization': f'Bearer {env.openai_key}',
                'Content-Type': 'application/json',
            },
            json={'model': 'tts-1', 'voice': 'alloy', 'input': 'Hello from Patter.'},
            timeout=30,
        )
        resp.raise_for_status()
        print(f'OpenAI TTS audio: {len(resp.content)} bytes')
        assert len(resp.content) > 0


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Calls a real number using ElevenLabs TTS in the Pipeline engine. Requires `ENABLE_LIVE_CALLS=1`.


### Pre-flight checklist


In [ ]:
with _setup.cell('live_preflight', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'ELEVENLABS_API_KEY'], env=env) as ok:
    if ok:
        print(f'  carrier:  Twilio {env.twilio_number}  →  {env.target_number}')
        print(f'  TTS:      ElevenLabs  voice={env.elevenlabs_voice_id[:8]}...')
        print(f'  webhook:  {env.public_webhook_url or "(ngrok auto-launch)"}')


### Live ElevenLabs TTS call *(T4)*


In [ ]:
from getpatter import Patter, Twilio, DeepgramSTT, OpenAILLM, ElevenLabsTTS
with _setup.cell('live_tts_call', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'ELEVENLABS_API_KEY', 'OPENAI_API_KEY'], env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid=env.twilio_sid, auth_token=env.twilio_token),
            phone_number=env.twilio_number,
            webhook_url=env.public_webhook_url,
        )
        agent = p.agent(
            system_prompt='Greet the caller with a short friendly message and end the call.',
            stt=DeepgramSTT(api_key=env.deepgram_key) if env.deepgram_key else None,
            llm=OpenAILLM(api_key=env.openai_key, model='gpt-4o-mini'),
            tts=ElevenLabsTTS(api_key=env.elevenlabs_key, voice_id=env.elevenlabs_voice_id),
        )
        try:
            await p.call(env.target_number, agent=agent,
                         first_message='Hello, this is a Patter ElevenLabs TTS demo.',
                         ring_timeout=env.max_call_seconds)
            print('✓ ElevenLabs TTS call completed')
        finally:
            _setup.hangup_leftover_calls(env)
